In [0]:
%sql
CREATE CATALOG IF NOT EXISTS openskyreal
MANAGED LOCATION 
'abfss://real@uralibootcamp.dfs.core.windows.net/managed/opensky';

CREATE SCHEMA IF NOT EXISTS openskyreal.bronze;

CREATE SCHEMA IF NOT EXISTS  openskyreal.silver;

CREATE SCHEMA IF NOT EXISTS  openskyreal.gold;

In [0]:
raw_path1 = "abfss://real@uralibootcamp.dfs.core.windows.net/raw/Flights"

checkpoint_path1 = "abfss://real@uralibootcamp.dfs.core.windows.net/checkpoint/Bronze_Flights"

schema_path1 = "abfss://real@uralibootcamp.dfs.core.windows.net/schema/Flights"
bronze_path1 = "abfss://real@uralibootcamp.dfs.core.windows.net/bronze/lights"

#bronze_table = "opensky.bronze.bronze_flights"

In [0]:
from pyspark.sql.types import *

opensky_schema1 = StructType([
    StructField("icao24", StringType(), True),
    StructField("callsign", StringType(), True),
    StructField("origin_country", StringType(), True),
    StructField("time_position", StringType(), True),
    StructField("longitude", StringType(), True),
    StructField("latitude", StringType(), True),
    StructField("baro_altitude", StringType(), True),
    StructField("velocity", StringType(), True),
    StructField("heading", StringType(), True),
    StructField("on_ground", StringType(), True)
])

Auto Loader batch ingestion

Add ingestion metadata

For Bronze history tracking:

In [0]:
from pyspark.sql.functions import *

bronze_df1 = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", schema_path1)
    .load(raw_path1)
    .withColumn(
        "source_file",
        col("_metadata.file_path")
    )
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
)

Write to Bronze Delta table (Real Time)
trigger(processingTime="1 minute") → Real-Time Streaming

Continuous micro-batch processing.


In [0]:
query1 = (
    bronze_df1.writeStream
    .format("delta")
    .option(
        "checkpointLocation",
        checkpoint_path1
    )
    .outputMode("append")
    .trigger(
        processingTime="2 minutes"
    )
    .table("openskyreal.bronze.bronze_flights")
)

In [0]:
query2 = (
    bronze_df1.writeStream
    .format("csv")
    .option("header", "true")
    .option("checkpointLocation", checkpoint_path1)
    .outputMode("append")
    .trigger(processingTime="2 minute")
    .option("path", bronze_path1)
    .start()
)